# Cost & Energy of Everyday LLM Workloads

This notebook runs the benchmark pipeline used for the PyCon poster. Start with the mock provider to verify the workflow, then switch to the Ollama config on a CPU/GPU VM.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ROOT

## 1. Smoke Test With The Mock Provider

The mock provider produces deterministic fake model timings so the plots and tables can be regenerated on any laptop.

In [ ]:
mock_out = ROOT / 'results' / 'notebook_mock'
subprocess.run([
    'python', str(ROOT / 'scripts' / 'run_benchmark.py'),
    '--config', str(ROOT / 'configs' / 'example_mock.yaml'),
    '--out', str(mock_out),
    '--limit', '2',
    '--repeats', '1',
], check=True)

metrics = pd.read_csv(mock_out / 'metrics.csv')
metrics.head()

In [ ]:
metrics.groupby(['workload', 'hardware_profile']).agg(
    requests=('item_id', 'count'),
    mean_latency_s=('total_latency_s', 'mean'),
    mean_cost_usd=('total_cost_usd', 'mean'),
    mean_energy_j=('total_energy_j', 'mean'),
).reset_index()

In [ ]:
subprocess.run([
    'python', str(ROOT / 'scripts' / 'make_plots.py'),
    '--metrics', str(mock_out / 'metrics.csv'),
    '--out', str(mock_out / 'figures'),
], check=True)
sorted((mock_out / 'figures').glob('*.png'))[:5]

## 2. Run The Real Ollama CPU/GPU Benchmark

On the VM, install Ollama, pull the model tags listed in `configs/example_ollama.yaml`, and flip `RUN_OLLAMA` to `True`.

In [ ]:
RUN_OLLAMA = False
ollama_out = ROOT / 'results' / 'ollama_vm'

if RUN_OLLAMA:
    subprocess.run([
        'python', str(ROOT / 'scripts' / 'run_benchmark.py'),
        '--config', str(ROOT / 'configs' / 'example_ollama.yaml'),
        '--out', str(ollama_out),
        '--repeats', '3',
    ], check=True)
    subprocess.run([
        'python', str(ROOT / 'scripts' / 'make_plots.py'),
        '--metrics', str(ollama_out / 'metrics.csv'),
        '--out', str(ollama_out / 'figures'),
    ], check=True)
    display(pd.read_csv(ollama_out / 'metrics.csv').head())
else:
    print('Set RUN_OLLAMA = True on the VM after pulling the configured models.')